> **Chapter 9, Part 3** | Sits between data quality and retrieval. **Focus:** master data, governance, stewardship, and the control layer that keeps enterprise semantics from fragmenting.


# Master Data Management and Governance

This notebook does one important job: it gives the later governance material something concrete to stand on.

Data quality alone is not enough. A dataset can pass validation checks and still fail at the enterprise level because customer records are duplicated, product hierarchies drift, reference codes fork, or ownership is vague when definitions conflict. MDM and governance exist to deal with those problems.

## Outputs

- a working distinction between master data, reference data, and transactional data
- a vocabulary for stewardship, decision rights, lineage, hierarchy control, and the golden record
- a simple duplicate-resolution example in Python
- a bridge into the advanced fractals and pattern-recognition module

## Supporting reading

- IBM on master data management: https://www.ibm.com/think/topics/master-data-management
- Abraham, Schneider, and vom Brocke (2019) on data governance: https://link.springer.com/article/10.1007/s12599-019-00588-3
- Hikmawati et al. (2021) review of MDM and governance: https://kinetik.umm.ac.id/index.php/kinetik/article/view/1272

## Failure note

If no one can tell you which customer record is authoritative, you do not have a governance problem later. You have one now.

## How I would debug this

I start with one business entity, one conflicting record cluster, one downstream report, and one owner. If those four things are still fuzzy, the process is decorative.


## Core distinctions

- **master data**: relatively stable entities the rest of the enterprise depends on, such as customer, supplier, product, location, and account
- **reference data**: controlled vocabularies and classification systems, such as country codes, currency codes, risk ratings, and product families
- **transactional data**: event-level records, such as orders, payments, claims, and trades

Governance is not synonymous with policy documents. In practice it means decision rights, standards, escalation paths, and evidence about how data should be created, changed, reconciled, and consumed.

MDM is one of the operational mechanisms that governance uses. It resolves duplicates, reconciles conflicting source records, stabilizes hierarchies, and produces an authoritative entity view.


In [ ]:
import pandas as pd

records = pd.DataFrame(
    [
        {"source": "crm", "customer_id": "C-100", "name": "Acme Health", "country": "US", "industry": "Healthcare", "updated_at": "2026-03-01"},
        {"source": "erp", "customer_id": "100", "name": "ACME Health Inc.", "country": "USA", "industry": "Health Care", "updated_at": "2026-03-05"},
        {"source": "support", "customer_id": "ACME-01", "name": "Acme Health", "country": "United States", "industry": "Healthcare", "updated_at": "2026-02-18"},
    ]
)

records["updated_at"] = pd.to_datetime(records["updated_at"])
records


In [ ]:
country_map = {
    "US": "US",
    "USA": "US",
    "United States": "US",
}

industry_map = {
    "Healthcare": "Healthcare",
    "Health Care": "Healthcare",
}

master = (
    records.assign(
        country_norm=records["country"].map(country_map),
        industry_norm=records["industry"].map(industry_map),
    )
    .sort_values("updated_at", ascending=False)
    .iloc[0]
)

master_record = {
    "golden_customer_name": "Acme Health",
    "canonical_country": master["country_norm"],
    "canonical_industry": master["industry_norm"],
    "surviving_source": master["source"],
    "surviving_customer_id": master["customer_id"],
}

pd.Series(master_record)


## What happened

The Python is simple because the point is conceptual.

We normalized reference values, chose survivorship logic, and produced a small golden record. Real MDM programs add match scoring, stewardship workflows, hierarchy governance, audit trails, exception queues, and source-system feedback. The logic stays the same: you cannot scale downstream analytics if the identity layer is unstable.

## Why this comes before the fractal topic

The advanced module asks whether defect patterns recur across scale. That question only becomes meaningful once learners understand what a governed entity, hierarchy, reference domain, or stewardship boundary actually is.

## Exercise

Pick one master entity class from your own work, or invent one if needed. List:

1. the canonical attributes you would preserve
2. the source systems you would trust least and most
3. the reference domains that would need active governance
4. the conditions that should trigger human stewardship rather than automatic merge
